# NRTK Tutorial

A guide to running NRTK via the Reference Implementation (RI).

## What is NRTK?

The Natural Robustness Toolkit (NRTK) is an open source toolkit for generating perturbations to images designed to mimic events that may be encountered in the real world.

See the [Official NRTK Documentation](https://nrtk.readthedocs.io/en/latest/) for the latest.

## Background and Overview

### NRTK Components
In the context of RI, these two NRTK interfaces are most relevant:

- **Perturbers** - Algorithms that modify an image based on a preconfigured set of rules.  For example, when using a brightness perturber with the factor set to 0.5, the brightness of the initial image will be halved.
- **Perturbation Factories** - A generator of *Perturbers* of a common class with input configurations varying along one or more dimensions. *Theta keys* are the parameters that will be changed and the *Thetas* are the values that will be in those parameters in each perturber.

### The RI NRTK Capability

In short, the NRTK RI capability works as follows:

* Specify one MAITE-compatible model, one dataset, one metric.  (You will also need to specify a threshold value for displaying results)
* Generate an NRTK Robustness capability configuration which specifies a *Perturbation Factory*, including:
    * A *Perturber* which specifies an algorithm.  For example *JitterOTFPerturber* applies an Optical Transfer Function (OTF) which simulates camera jitter.
    * A *Perturbation Factory* which specifies one or more `theta_keys` and any additional parameters made available by the implementation.  In the example that follows, the perturber factory accepts one theta key, as well as `start`, `stop`, and `step`.  (See definitions below.)
* Execute the capability, which will:
    * Apply each yielded perturbation (i.e. possible combination of values for `theta_keys`) to the dataset
    * Run the configured metric against each perturbed dataset
    * Generate a report that displays the change in the model performance (as measured by the provided metric) as each `theta_key` changes within its specified range.

Keep in mind these two considerations when using the `NrtkRobustness`.

1) Be aware of performance considerations.  For each perturber configuration, the `NrtkRobustness` capability must perturb and generate predictions over the entire dataset, and that the number of perturber configurations is a product of all of the possible theta values.  For example, if three *theta keys* are specified and each key takes five values, the capability will generate `5*5*5=125` predictions *across the entire dataset*.

2) `NrtkRobustness` does NOT save the image perturbations - only the metrics resulting from the *theta key* changes.

## Demonstration

In plain language, we will be measuring how a model accuracy degrades at five levels of increasing camera jitter.

This is the `NrtkRobustness` capability configuration which we will use as our baseline as we demonstrate the features of the NRTK stack.

```
{
    "name": "natural_robustness_jitter_model_and_sim",
    "perturber_factory": {
        "type": "nrtk.impls.perturb_image_factory.generic.linspace_step.LinSpacePerturbImageFactory",
        "nrtk.impls.perturb_image_factory.generic.linspace_step.LinSpacePerturbImageFactory": {
            "perturber": "nrtk.impls.perturb_image.pybsm.jitter_otf_perturber.JitterOTFPerturber",
            "theta_key": "s_x",
            "start": 0.0,
            "stop": 0.00015,
            "step": 5
        }
}
```

We will use a [LinSpacePerturbImageFactory](https://nrtk.readthedocs.io/en/latest/_implementations/nrtk.impls.perturb_image_factory.generic.linspace_step.LinSpacePerturbImageFactory.html) which generates a linear distribution of perturbations across the provided range. It has the following relevant configurations.

* **perturber** – Python implementation type of the PerturbImage interface to produce.
* **theta_key**  – Perturber parameter to vary between instances.
* **start** – Initial value of desired range (inclusive).
* **stop** – Final value of desired range (exclusive).
* **step** – Number of instances to generate.



First, we create the three necessary MAITE-wrapped objects - a *dataset*, a compatible *model*, and a *metric*.

First, a `VisdroneDetectionDataset` wrapper is generated using data found in our test directory, a four-image subset of the Visdrone test dataset.

Then a `VisdroneODModel` wrapper with `resnet18` architecture is generated using a ResNet model with weights pre-trained by Kitware, Inc.

Finally, the default `map50_torch_metric_factory` function is used to generate a mean average precision metric to be used for evaluating performance.

In [ ]:
from pathlib import Path
from jatic_ri.core.object_detection.dataset_loaders import VisdroneDetectionDataset
from jatic_ri.core.object_detection.models import VisdroneODModel
from jatic_ri.core.object_detection.metrics import map50_torch_metric_factory

BASE_DIR = Path.cwd().parents[1]
dataset_root_path = BASE_DIR / "tests/data_for_tests/visdrone_dataset"
dataset_ann_file_path = BASE_DIR / "tests/data_for_tests/visdrone_dataset/ann_file.json"
model_name = "resnet18"

print("Loading Visdrone dataset...")
# Use the example Visdrone dataset
dataset = VisdroneDetectionDataset(root=dataset_root_path)
print(f"Dataset loaded with {len(dataset)} images")

model = VisdroneODModel(arch=model_name)

metric = map50_torch_metric_factory()
metric.metadata["id"] = metric.return_key

Next, we initialize an `NrtkRobustness` object.  We load the dataset, model, and metric wrapped above.  We also load a threshold which will become an input for visualizations in the reports rendered by this capability.  In the example below we have arbitrarily chosen 0.15, arbitrarily selected to appear relevant next to the actual metric outputs).

After loading all the necessary stage inputs, we run the tool.

In [ ]:
from jatic_ri.core.object_detection import NrtkRobustness, NrtkRobustnessConfig

nrtk_config_dict = {
    "name": "natural_robustness_jitter_model_and_sim",
    "perturber_factory": {
        "type": "nrtk.impls.perturb_image_factory.generic.linspace_step.LinSpacePerturbImageFactory",
        "nrtk.impls.perturb_image_factory.generic.linspace_step.LinSpacePerturbImageFactory": {
            "perturber": "nrtk.impls.perturb_image.pybsm.jitter_otf_perturber.JitterOTFPerturber",
            "theta_key": "s_x",
            "start": 0.0,
            "stop": 0.00015,
            "step": 5
        },
    },
}
nrtk_config = NrtkRobustnessConfig(**nrtk_config_dict)
capabililty = NrtkRobustness()
output = capabililty.run(use_cache=False, datasets=[dataset], models=[model], metrics=[metric], config=nrtk_config)

## Slide Deck

Once the run has completed, the code below uses the `gradient` package to create HTML and PPTX formatted reports of the results of the NRTK capability.

In [ ]:
import os
from gradient.templates_and_layouts.create_deck import create_deck

output_dir = Path("nrtk_example_output")
os.makedirs(output_dir, exist_ok=True)

create_deck(output.collect_report_consumables(threshold=0.15), output_dir, deck_name='NRTK_Example_Report')
print(f"PowerPoint saved in {output_dir}.")